In [ ]:
from helpers import *
import require


folder = "10991360/"

pd.set_option('display.max_rows', None) 

csv_df = pd.read_csv('customers_sales.csv')

csv_df['customer_name'] = csv_df['customer_name'].replace({
    'AutoPart AG': 'AutoParts AG',
    'Metalform SA': 'MetalForm SA',
    'PrecisionCut GmbH ': 'PrecisionCut GmbH',   # Leerzeichen am Ende
})

#csv_df = csv_df.drop_duplicates(ignore_index=True) #Annahmen: Doppelte Zeilen sind absicht -> Auskommentiert

fehlend = sorted(set(os.listdir(folder)) - set(csv_df['file_name']))
neu = pd.DataFrame({'file_name': fehlend, 'order_quantity': 0})
csv_df = pd.concat([csv_df, neu], ignore_index=True)

csv_df["id"] = csv_df.groupby(["customer_name", "customer_branch"], sort=False, dropna=False).ngroup() + 1

print(csv_df['customer_name'].value_counts())
print(csv_df['customer_branch'].value_counts())

csv_df = csv_df.sort_values(by="id")

display(csv_df)

csv_df.to_csv('data_preparation_output.csv', index=False)

NameError: name 'os' is not defined

In [13]:
cutting_speed_mm_min = 1000 # mm/min
offset_general_s = 120 #s 
offset_time_smartgate_s = 2 #s
offset_time_sortmaster_s = 5 #s
offset_time_manually_s = 20 #s
offset_time_hole_s = 3 #s

smartgate_min = (30, 30) # min
smartgate_max = (160, 160) # max

sortmaster_min = (60, 90) # min
sortmaster_max = (1500, 2000) # max

def get_cutting_time(path):
    cutting_time_s = get_cutting_length(path) / (cutting_speed_mm_min / 60)

    extraction_method = get_extraction_method(path)

    if extraction_method == "Gate + Master":
        offset_extraction_method_s = min(offset_time_smartgate_s, offset_time_sortmaster_s)
    elif extraction_method == "Master":
        offset_extraction_method_s = offset_time_sortmaster_s
    elif extraction_method == "Gate":
        offset_extraction_method_s = offset_time_smartgate_s
    else:
        offset_extraction_method_s = offset_time_manually_s

    offset_time_total_s = offset_general_s + offset_extraction_method_s + get_number_of_holes(path) * offset_time_hole_s

    return cutting_time_s + offset_time_total_s

def get_fit(object, min_opening, max_opening):
    ob_1, ob_2 = sorted(object)
    min_1, min_2 = sorted(min_opening)
    max_1, max_2 = sorted(max_opening)

    return min_1 <= ob_1 <= max_1 and min_2 <= ob_2 <= max_2

def get_extraction_method(path):

    bounding_box = tuple(sorted(get_dxf_bound(path)))

    if get_fit(bounding_box, sortmaster_min, sortmaster_max) and get_fit(bounding_box, smartgate_min, smartgate_max):
        return "Gate + Master"
    
    elif get_fit(bounding_box, smartgate_min, smartgate_max):
        return "Gate"
    
    elif get_fit(bounding_box, sortmaster_min, sortmaster_max):
        return "Master"
    
    else:
        return "Manual"

def get_bounding_box_area(path):
    bounding_box = get_dxf_bound(path)
    return bounding_box[0] * bounding_box[1]

def get_automation_index(extraction_method):

    if extraction_method != "Manual":
        return 1
    else:
        return 0

csv_df['path'] = folder + csv_df['file_name']

csv_df[['bounding_box_width_mm', 'bounding_box_height_mm']] = csv_df['path'].apply(lambda p: pd.Series(get_dxf_bound(p)))

csv_df['bounding_box_area_mm^2'] = csv_df['bounding_box_width_mm'] * csv_df['bounding_box_height_mm']
csv_df['part_area_mm^2'] = csv_df['path'].apply(get_part_area)
csv_df['aspect_ratio'] = csv_df['bounding_box_height_mm'] / csv_df['bounding_box_width_mm']
csv_df['rectangularity_score'] = csv_df['part_area_mm^2'] / csv_df['bounding_box_area_mm^2']

csv_df['extraction_method'] = csv_df["path"].apply(get_extraction_method)
csv_df['automation_index'] = csv_df["extraction_method"].map(get_automation_index)

csv_df['number_of_holes'] = csv_df["path"].apply(get_number_of_holes)
csv_df['cutting_length_mm'] = csv_df["path"].apply(get_cutting_length)
csv_df['cutting_time_s'] = csv_df["path"].apply(get_cutting_time)

In [14]:
parts_df = csv_df.copy()
gruppe = parts_df.groupby('file_name')
parts_df['total_sales'] = gruppe['order_quantity'].transform('sum')
parts_df['number_of_sales'] = gruppe['order_quantity'].transform(lambda s: (s > 0).sum())

csv_df_parts_sorted = (
    parts_df.drop(columns=['order_quantity', 'customer_name', 'customer_branch'])
    .drop_duplicates('file_name')
    .sort_values('file_name', key=lambda s: s.str.extract(r'\((\d+)\)', expand=False).astype(int), ignore_index=True)
)
display(csv_df_parts_sorted)

,file_name,id,path,bounding_box_width_mm,bounding_box_height_mm,bounding_box_area_mm^2,part_area_mm^2,aspect_ratio,rectangularity_score,extraction_method,automation_index,number_of_holes,cutting_length_mm,cutting_time_s,total_sales,number_of_sales
0,Geo (1).DXF,1,10991360/Geo (1).DXF,1600.219048,305.139765,4.882905e+05,4.013592e+04,0.190686,0.082197,Master,1,0,3438.024977,331.281499,1600,1
1,Geo (2).DXF,4,10991360/Geo (2).DXF,1274.000000,38.643216,4.923146e+04,3.374337e+04,0.030332,0.685403,Manual,0,0,2698.517849,301.911071,8250,1
2,Geo (3).DXF,1,10991360/Geo (3).DXF,1459.852300,400.302700,5.843828e+05,3.428389e+05,0.274208,0.586668,Master,1,0,7502.466477,575.147989,1900,1
3,Geo (4).DXF,2,10991360/Geo (4).DXF,286.897900,164.397900,4.716541e+04,1.666551e+04,0.573019,0.353342,Master,1,9,1380.000951,234.800057,9400,1
4,Geo (5).DXF,3,10991360/Geo (5).DXF,170.565455,170.565455,2.909257e+04,9.259805e+03,1.000000,0.318288,Master,1,14,1260.659996,242.639600,5000,1
5,Geo (6).DXF,3,10991360/Geo (6).DXF,201.666187,69.095365,1.393420e+04,1.243090e+04,0.342622,0.892114,Master,1,6,809.082784,191.544967,4000,1
6,Geo (7).DXF,2,10991360/Geo (7).DXF,374.625306,17.386985,6.513605e+03,6.058875e+03,0.046412,0.930188,Manual,0,3,942.159585,205.529575,9500,1
7,Geo (8).DXF,5,10991360/Geo (8).DXF,801.705747,344.180571,2.759315e+05,1.186345e+05,0.429310,0.429942,Master,1,15,6111.748725,536.704923,12050,1
8,Geo (9).DXF,3,10991360/Geo (9).DXF,209.773263,158.699000,3.329081e+04,1.857517e+04,0.756526,0.557967,Master,1,6,994.390298,202.663418,3750,1
9,Geo (10).DXF,2,10991360/Geo (10).DXF,853.830303,333.401400,2.846682e+05,2.061794e+05,0.390477,0.724280,Master,1,5,3217.207505,333.032450,9650,1


In [15]:
csv_df.to_csv('parts_and_customer.csv', index=False)
csv_df_parts_sorted.to_csv('parts.csv', index=False)